In [31]:
import pandas as pd
import numpy as np
import requests

!pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
books = pd.read_csv("C:/Users/jb24d625/OneDrive/Doktorat/CAS/Modul 6/book_recommender/second dataset/books_1.Best_Books_Ever.csv")

In [33]:
print(books.head())
books

                                        bookId  \
0                     2767052-the-hunger-games   
1  2.Harry_Potter_and_the_Order_of_the_Phoenix   
2                   2657.To_Kill_a_Mockingbird   
3                     1885.Pride_and_Prejudice   
4                               41865.Twilight   

                                       title                 series  \
0                           The Hunger Games    The Hunger Games #1   
1  Harry Potter and the Order of the Phoenix        Harry Potter #5   
2                      To Kill a Mockingbird  To Kill a Mockingbird   
3                        Pride and Prejudice                    NaN   
4                                   Twilight   The Twilight Saga #1   

                                      author  rating  \
0                            Suzanne Collins    4.33   
1  J.K. Rowling, Mary GrandPré (Illustrator)    4.50   
2                                 Harper Lee    4.28   
3  Jane Austen, Anna Quindlen (Introduction)    

,bookId,title,series,author,rating,description,language,isbn,genres,characters,...,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",...,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,English,9780439358071,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...","['Sirius Black', 'Draco Malfoy', 'Ron Weasley'...",...,06/21/03,['Bram Stoker Award for Works for Young Reader...,2507623,"['1593642', '637516', '222366', '39573', '14526']",98.0,['Hogwarts School of Witchcraft and Wizardry (...,https://i.gr-assets.com/images/S/compressed.ph...,2632233,26923,7.38
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,English,9999999999999,"['Classics', 'Fiction', 'Historical Fiction', ...","['Scout Finch', 'Atticus Finch', 'Jem Finch', ...",...,07/11/60,"['Pulitzer Prize for Fiction (1961)', 'Audie A...",4501075,"['2363896', '1333153', '573280', '149952', '80...",95.0,"['Maycomb, Alabama (United States)']",https://i.gr-assets.com/images/S/compressed.ph...,2269402,23328,NaN
3,1885.Pride_and_Prejudice,Pride and Prejudice,NaN,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,English,9999999999999,"['Classics', 'Fiction', 'Romance', 'Historical...","['Mr. Bennet', 'Mrs. Bennet', 'Jane Bennet', '...",...,01/28/13,[],2998241,"['1617567', '816659', '373311', '113934', '767...",94.0,"['United Kingdom', 'Derbyshire, England (Unite...",https://i.gr-assets.com/images/S/compressed.ph...,1983116,20452,NaN
4,41865.Twilight,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,English,9780316015844,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...","['Edward Cullen', 'Jacob Black', 'Laurent', 'R...",...,10/05/05,"['Georgia Peach Book Award (2007)', 'Buxtehude...",4964519,"['1751460', '1113682', '1008686', '542017', '5...",78.0,"['Forks, Washington (United States)', 'Phoenix...",https://i.gr-assets.com/images/S/compressed.ph...,1459448,14874,2.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52473,11492014-fractured,Fractured,Fateful #2,Cheri Schmidt (Goodreads Author),4.00,The Fateful Trilogy continues with Fractured. ...,English,2940012616562,"['Vampires', 'Paranormal', 'Young Adult', 'Rom...",[],...,NaN,[],871,"['311', '310', '197', '42', '11']",94.0,[],https://i.gr-assets.com/images/S/compressed.ph...,0,1,NaN
52474,11836711-anasazi,Anasazi,Sense of Truth #2,Emma Michaels,4.19,"'Anasazi', sequel to 'The Thirteenth Chime' by...",English,9999999999999,"['Mystery', 'Young Adult']",[],...,August 3rd 2011,[],37,"['16', '14', '5', '2', '0']",95.0,[],https://i.gr-assets.com/images/S/compressed.ph...,0,1,NaN
52475,10815662-marked,Marked,Soul Guardians #1,Kim Richardson (Goodreads Author),3.70,--READERS FAVORITE AWARDS WINNER 2011--Sixteen...,English,9781461017097,"['Fantasy', 'Young Adult', 'Paranormal', 'Ange...",[],...,March 15th 2011,"[""Readers' Favorite Book Award (2011)""]",6674,"['2109', '1868', '1660', '647', '390']",84.0,[],https://i.gr-assets.com/images/S/compressed.ph...,0,1,7.37
52476,11330278-wayward-son,Wayward Son,NaN,"Tom Pollack (Goodreads Author), John Loftus (G...",3.85,A POWERFUL TREMOR UNEARTHS AN ANCIENT SECRETBu...,English

In [34]:
#remove unnecessary columns
books = books.drop(
    columns = [
            "bookId", 
            "isbn","language", 
            "characters", 
            "firstPublishDate", 
            "awards", 
            "setting", 
            "coverImg",	
            "bbeScore", 
            "bbeVotes",	
            "price",
            "bookFormat",
            "edition",
            "pages",
            "publisher",
            "publishDate"
        ]
    )

In [35]:
print(books.columns)
books.dtypes

Index(['title', 'series', 'author', 'rating', 'description', 'genres',
       'numRatings', 'ratingsByStars', 'likedPercent'],
      dtype='object')


title              object
series             object
author             object
rating            float64
description        object
genres             object
numRatings          int64
ratingsByStars     object
likedPercent      float64
dtype: object

In [36]:
print(type(books['description'].iloc[0]))

<class 'str'>


In [37]:
#check missing data
books.isnull()

,title,series,author,rating,description,genres,numRatings,ratingsByStars,likedPercent
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False
3,False,True,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
52473,False,False,False,False,False,False,False,False,False
52474,False,False,False,False,False,False,False,False,False
52475,False,False,False,False,False,False,False,False,False
52476,False,True,False,False,False,False,False,False,False


In [38]:
#rename columns for cleaner code
books.rename(columns={
    'numRatings': 'numratings',
    'ratingsByStars': 'ratingsbystars',
    'likedPercent': 'likedpercent'
    }, inplace=True)
print(books.columns)
#sanity check
books.to_csv('cleaned_books.csv', index=False)

Index(['title', 'series', 'author', 'rating', 'description', 'genres',
       'numratings', 'ratingsbystars', 'likedpercent'],
      dtype='object')


In [39]:
#NLP recommender needs one piece of text per book. Therefore I create a profile for each book in which the important
#text passages are included
books['bookprofile'] = (
    books['title'].fillna('') + ' ' +
    books['author'].fillna('') + ' ' +
    books['genres'].fillna('') + ' ' +
    books['description'].fillna('')
)


In [40]:
#with this model every book becomes a semantic vector
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    books['bookprofile'].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1640 [00:00<?, ?it/s]

In [42]:
#cosine similarity
cosine_sim = cosine_similarity(embeddings, embeddings)

In [43]:
#weighted rating for a hybrid recommender system
#weighting penalized low-review books and rewards consistent popularity
C = books['rating'].mean()
m = books['numratings'].quantile(0.75)

def weighted_rating(x, m=m, C=C): 
    v = x['numratings'] 
    R = x['rating'] 
    return (v / (v + m)) * R + (m / (v + m)) * C 
    
books['score'] = books.apply(weighted_rating, axis=1)

In [44]:
#first you find the index of the book you want
#then you get the similarity scores and sort them with the most similar books first

indices = pd.Series(
    books.index,
    index=books['title']
).drop_duplicates()

def hybrid_recommend(title, top_n=10):

    if title not in indices: 
        return f"books '{title}' not found." 

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx])) 
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:50]
    book_indices = [i[0] for i in sim_scores]
    candidates = books.iloc[book_indices].copy()
    candidates['similarity'] = [i[1] for i in sim_scores]
    candidates['norm_score'] = candidates['score'] / candidates['score'].max()
    candidates['final_score'] = (
        0.7 * candidates['similarity'] +
        0.3 * candidates['norm_score']
    )

    candidates = candidates.sort_values('final_score', ascending=False)

    return candidates[['title', 'author', 'genres', 'description', 'rating', 'numratings']].head(top_n)

In [45]:
#test
book_name = "Fantastic Beasts and Where to Find Them: The Original Screenplay"
print("\nRecommendations:\n") 
print(hybrid_recommend(book_name))


Recommendations:

                                                   title  \
1184             Fantastic Beasts and Where to Find Them   
9637   Fantastic Beasts - The Crimes of Grindelwald: ...   
51722       Hogwarts: An Incomplete and Unreliable Guide   
7418                         Harry Potter: Film Wizardry   
2286                             Harry Potter Collection   
13472                               The Hogwarts Library   
409                          Harry Potter Series Box Set   
1600                     The Harry Potter Collection 1-4   
40815  The Magical Worlds Of Harry Potter: A Treasury...   
7008   Harry Potter Boxed Set, Books 1-5 (Harry Potte...   

                                          author  \
1184    Newt Scamander (Pseudonym), J.K. Rowling   
9637                                J.K. Rowling   
51722                               J.K. Rowling   
7418                                Brian Sibley   
2286                                J.K. Rowling   
13472   

In [46]:
print("Number of books:", len(books))
print("Similarity matrix shape:", cosine_sim.shape)

Number of books: 52478
Similarity matrix shape: (52478, 52478)


Semantic Retrieval System

In [47]:
#model so user queries can be converted into embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [48]:
#create a rating score 
C = books['rating'].mean()
m = books['numratings'].quantile(0.75)

def weighted_rating(x):

    v = x['numratings']
    R = x['rating']
    return ((v / (v + m)) * R + (m / (v + m)) * C)

books['score'] = books.apply(
    weighted_rating,
    axis=1
)

In [49]:
#semantic retrieval function
def semantic_search(query, top_n=10):

    query_embedding = model.encode([query])
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    candidates = books.copy()
    candidates['similarity'] = similarities
    candidates['norm_score'] = (
        candidates['score']
        /
        candidates['score'].max()
    )

    candidates['final_score'] = (0.8 * candidates['similarity'] + 0.2 * candidates['norm_score'])
    candidates = candidates.sort_values(
        'final_score',
        ascending=False
    )

    return candidates[
        [
            'title',
            'author',
            'rating',
            'numratings',
            'similarity',
            'final_score'
        ]
    ].head(top_n)

In [50]:
#example:
print(semantic_search("a fantasy book with dragons and a female main character"))

                              title  \
31268     Dragonsong / Dragonsinger   
6216   A Natural History of Dragons   
22757             Loved by a Dragon   
10592    Dragonflight / Dragonquest   
2549       The Dragonriders of Pern   
1978     Dragons of Autumn Twilight   
25146   Dragon Lyric: A Short Story   
13355                 Dragon Keeper   
47302             Playing With Fire   
11905         Searching for Dragons   

                                                  author  rating  numratings  \
31268                                     Anne McCaffrey    4.56        1930   
6216                    Marie Brennan (Goodreads Author)    3.80       25894   
22757                Linda K. Hopkins (Goodreads Author)    4.25         459   
10592  Anne McCaffrey, Adrienne Barbeau (Read by), Mi...    4.47       10906   
2549                                      Anne McCaffrey    4.24       27108   
1978   Margaret Weis (Goodreads Author), Tracy Hickma...    3.99      108100   
25146     

Now add additional books via Google API to get books that are not in the dataset (dataset is at least 2 years old)

In [51]:
queries = [
    "fantasy",
    "romance",
    "science fiction",
    "thriller",
    "mystery",
    "new adult"
]

new_books = []

key = "ADD YOUR API KEY"

for query in queries:

    url = (
        "https://www.googleapis.com/books/v1/volumes"
        f"?q={query}"
        "&maxResults=40"
        "&key=" + key
    )

    response = requests.get(url)
    data = response.json()

for item in data.get("items", []):

        info = item.get("volumeInfo", {})

        new_books.append({
            "title": info.get("title", ""),
            "author": ", ".join(info.get("authors", [])),
            "description": info.get("description", ""),
            "genres": ", ".join(info.get("categories", [])),
            "publishedDate": info.get("publishedDate", "")
        })

new_books_df = pd.DataFrame(new_books)

#remove books that don't have a description
new_books_df = new_books_df.dropna(
    subset=["description"]
)

new_books_df = new_books_df[
    new_books_df["description"].str.len() > 50
]

print("Books fetched:", len(new_books_df))

Books fetched: 3


In [52]:
#Google API allows up to 40 books per request. This time 120 books were fetched, but only 3 had a description to work with. 
#As an addition to the 52.000 books that are in the dataset, this makes no sense as the recommender is not meaningfully expanded
#however, we know now that it works and can narrow down the use of the API expansion
#we want to add books from a specific genre to get books for my taste and add books from after the dataset was made

In [53]:
#new import, this time with books that were published >= 2023
#type of books are narrowed down for hopefully more and better results
#indexes are added so that multiple pages are fetched 

API_KEY = "ADD YOUR API KEY"

queries = [
    "romantasy",
    "high fantasy",
    "dragons",
    "wyvern",
    "war college",
    "new adult"
]

new_books = []

for query in queries:
    for start in range(0, 200, 40):
        url = (
            "https://www.googleapis.com/books/v1/volumes"
            f"?q={query}"
            f"&startIndex={start}"
            "&maxResults=40"
            f"&key={API_KEY}"
        )

        response = requests.get(url)
        data = response.json()

        for item in data.get("items", []):

            info = item.get("volumeInfo", {})

            new_books.append({
                "title": info.get("title", ""),
                "author": ", ".join(info.get("authors", [])),
                "description": info.get("description", ""),
                "genres": ", ".join(info.get("categories", [])),
                "publishedDate": info.get("publishedDate", "")
            })

new_books_df = pd.DataFrame(new_books)

print("Books fetched:", len(new_books_df))
print(new_books_df.head())

#remove books that don't have a description
new_books_df = new_books_df.dropna(
    subset=["description"]
)

new_books_df = new_books_df[
    new_books_df["description"].str.len() > 50
]

print("Books fetched:", len(new_books_df))

Books fetched: 600
                                               title          author  \
0                                          Romantasy    Isabella Mey   
1  Dead Souls: Die spicy Romantasy-Trilogie in ei...      Izzy Maxen   
2                            Infernas 1: King of Ash    Melanie Lane   
3                                     I Am the Blade  Marie Graßhoff   
4                                The dark between us    Lily Cattens   

                                         description   genres publishedDate  
0  Romantasy ... ist ein Inselkönigreich auf dem ...  Fiction    2024-02-28  
1  »Ich darf ihn nicht wiedersehen wollen. Ich da...  Fiction    2024-11-07  
2  »Und?«, raunte Dante leise, den Blick auf mein...  Fiction    2023-11-24  
3  Dieses Buch gibt es in zwei Versionen: mit und...  Fiction    2026-06-29  
4                                                                      2026  
Books fetched: 193


In [57]:
#create new bookprofiles for the new dataset
new_books_df['bookprofile'] = (
    books['title'].fillna('') + ' ' +
    books['author'].fillna('') + ' ' +
    books['genres'].fillna('') + ' ' +
    books['description'].fillna('')
)

In [59]:
#create combined dataset
books2 = pd.concat(
    [books, new_books_df],
    ignore_index=True
)

In [63]:
#create embeddings for the combined dataset

model = SentenceTransformer("all-MiniLM-L6-v2")
new_embeddings = model.encode(
    books2["bookprofile"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1646 [00:00<?, ?it/s]

In [60]:
#add the embeddings so that my retrieval system knows about the goodreads and google data
embeddings = np.vstack([embeddings, new_embeddings])

In [67]:
#save everything
books2.to_pickle("books2_semantic.pkl")
np.save("book2_embeddings.npy",new_embeddings)
print("new dataset saved.")

new dataset saved.


In [71]:
#semantic retrieval function for new dataset
def semantic_search(query, top_n=10):

    query_embedding = model.encode([query])
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    candidates = books2.copy()
    candidates['similarity'] = similarities
    candidates['norm_score'] = (
        candidates['score']
        /
        candidates['score'].max()
    )

    candidates['final_score'] = (0.8 * candidates['similarity'] + 0.2 * candidates['norm_score'])
    candidates = candidates.sort_values(
        'final_score',
        ascending=False
    )

    return candidates[
        [
            'title',
            'author',
            'rating',
            'numratings',
            'similarity',
            'final_score'
        ]
    ].head(top_n)

In [72]:
#example:
print(semantic_search("a fantasy book with dragons and a female main character"))

                              title  \
31268     Dragonsong / Dragonsinger   
6216   A Natural History of Dragons   
22757             Loved by a Dragon   
10592    Dragonflight / Dragonquest   
2549       The Dragonriders of Pern   
1978     Dragons of Autumn Twilight   
25146   Dragon Lyric: A Short Story   
13355                 Dragon Keeper   
47302             Playing With Fire   
11905         Searching for Dragons   

                                                  author  rating  numratings  \
31268                                     Anne McCaffrey    4.56      1930.0   
6216                    Marie Brennan (Goodreads Author)    3.80     25894.0   
22757                Linda K. Hopkins (Goodreads Author)    4.25       459.0   
10592  Anne McCaffrey, Adrienne Barbeau (Read by), Mi...    4.47     10906.0   
2549                                      Anne McCaffrey    4.24     27108.0   
1978   Margaret Weis (Goodreads Author), Tracy Hickma...    3.99    108100.0   
25146     

In [73]:
#example dataset books2:
print(semantic_search("a romantasy new adult book with dragons and a female main character"))

                              title  \
37738             A Gift of Dragons   
31268     Dragonsong / Dragonsinger   
2913           Dealing with Dragons   
13355                 Dragon Keeper   
6216   A Natural History of Dragons   
2549       The Dragonriders of Pern   
10592    Dragonflight / Dragonquest   
43513                   Dragonflame   
33951           Space Dragon Poetry   
22757             Loved by a Dragon   

                                                  author  rating  numratings  \
37738             Anne McCaffrey, Tom Kidd (Illustrator)    4.19      5747.0   
31268                                     Anne McCaffrey    4.56      1930.0   
2913   Patricia C. Wrede (Goodreads Author), Peter de...    4.13     84613.0   
13355                                   Carole Wilkinson    3.99     10633.0   
6216                    Marie Brennan (Goodreads Author)    3.80     25894.0   
2549                                      Anne McCaffrey    4.24     27108.0   
10592  Ann